# ASB Case Study 7C - Building a RAG and AI Agent with ShopEase Philippines Data

- Build Retrieval-Augmented Generation (RAG) pipeline to handle unstructured company policies
- Build tools to query live transactional tables
- Connect both to an AI Agent capable of multi-step reasoning

In [1]:
from dataclasses import dataclass
from pathlib import Path
from pprint import pprint
from collections.abc import Callable, Iterable
import inspect
import json

import docx
from bs4 import BeautifulSoup
from pypdf import PdfReader

import ollama
import numpy as np
import pandas as pd

In [2]:
DATA_DIR = Path().cwd() / "data"
EMBED_MODEL = "nomic-embed-text"
CHAT_MODEL = "llama3.2"

In [3]:
DATA_DIR

WindowsPath('D:/Users/Esperine/Workspace3/asb-main/cs7/cs7c/data')

## 1 - Data triage

| Category | Files | Best Access Pattern | Why? |
| --- | --- | --- | --- |
| Unstructured Knowledge (or Data) | `company_handbook.pdf`, `return_refund_policy.docx`, `shipping_policy.md`, `promotions.txt`, `faq.html` | RAG (Vector Search) | Written in natural language paragraphs; answers "how" and "why" policy questions. |
| Structured/Transactional Knowledge (or Data) | `orders.xlsx`, `product_catalog.csv`, `customers.json`, `stores.csv`, `store_inventory.csv`, `support_tickets.json` | Agent Tools (Code/SQL) | Tabular rows linked by IDs (sku, order_id, customer_id); changes every minute and requires exact lookups. |

## 2 - RAG (Unstructured knowledge)

### 2.1 - Data ingestion

In [4]:
def _parse_text(path: Path) -> str:
    return path.read_text(encoding="utf-8")


def _parse_pdf(path: Path) -> str:
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def _parse_docx(path: Path) -> str:
    doc = docx.Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())


def _parse_html(path: Path) -> str:
    soup = BeautifulSoup(path.read_text(encoding="utf-8"), "html.parser")
    return soup.get_text(separator="\n", strip=True)


PARSERS = {
    ".txt": _parse_text,
    ".md": _parse_text,
    ".pdf": _parse_pdf,
    ".docx": _parse_docx,
    ".html": _parse_html,
}


def load_knowledge_docs(
    filenames: Iterable[str], 
    data_dir: str | Path = "./data"
) -> list[dict]:
    """Loads and extracts text from an iterable of filenames."""
    base_path = Path(data_dir)
    documents = []

    for file in filenames:
        file_path = base_path / file

        if not file_path.exists():
            print(f"[Warning] Skipping missing file: {file_path}")
            continue

        ext = file_path.suffix.lower()
        parser = PARSERS.get(ext)

        if not parser:
            print(f"[Warning] Unsupported format '{ext}' for {file_path.name}")
            continue

        documents.append({
            "source": file_path.name,
            "text": parser(file_path)
        })

    return documents

In [5]:
knowledge_files = [
    "company_handbook.pdf",
    "return_refund_policy.docx",
    "shipping_policy.md",
    "promotions.txt",
    "faq.html"
]

raw_docs = load_knowledge_docs(knowledge_files, data_dir=DATA_DIR)
raw_docs[0]

{'source': 'company_handbook.pdf',
 'text': 'ShopEase Philippines Employee Handbook\nVersion 1.0 - Effective 1 August 2026\n1. Company Overview\nShopEase Philippines is a fictional omnichannel retail company serving customers through online ordering and\nphysical stores across the Philippines. This handbook is designed as synthetic training material for an analytics\nand AI exercise.\n2. Working Hours\nCorporate office hours are 9:00 AM to 6:00 PM, Monday through Friday. Store employees follow schedules based\non branch operating hours and approved shifts.\n3. Annual Leave\nRegular full-time employees receive 15 days of paid annual leave per calendar year. Up to 5 unused annual leave\ndays may be carried over to the next calendar year.\n4. Sick Leave\nRegular employees receive 10 days of paid sick leave per calendar year. A medical certificate is required for\nabsences of 3 or more consecutive working days.\n5. Work From Home\nEligible corporate employees may work remotely up to 2 days

### 2.2 - Chunking and vector indexing

In [6]:
@dataclass
class Chunk:
    source: str
    chunk_id: int
    text: str
    embedding: np.ndarray | None = None


def chunk_documents(
    docs: Iterable[dict], 
    chunk_size: int = 500, 
    overlap: int = 50
) -> list[Chunk]:
    """Splits extracted documents into overlapping Chunk objects with metadata."""
    
    if overlap >= chunk_size:
        raise ValueError("overlap must be strictly smaller than chunk_size")

    chunks: list[Chunk] = []
    step = chunk_size - overlap
    for d in docs:
        text = d["text"]
        for chunk_id, start in enumerate(range(0, len(text), step)):
            snippet = text[start:start + chunk_size].strip()
            if snippet:
                chunks.append(
                    Chunk(
                        source=d["source"],
                        chunk_id=chunk_id,
                        text=snippet
                    )
                )

    return chunks


def get_embedding(text: str, model: str = EMBED_MODEL) -> np.ndarray:
    response = ollama.embeddings(model=model, prompt=text)
    return np.asarray(response["embedding"], dtype=np.float64)


def index_chunks(
    chunks: list[Chunk], 
    model: str = EMBED_MODEL
) -> tuple[list[Chunk], np.ndarray]:
    """Computes L2-normalized embeddings and stacks them into a 2D search matrix."""
    vectors = []
    for c in chunks:
        vec = get_embedding(c.text, model=model)
        norm = np.linalg.norm(vec)
        c.embedding = vec / norm if norm > 0 else vec
        vectors.append(c.embedding)

    # Shape - (num_chunks, embedding_dim)
    embedding_matrix = np.vstack(vectors)
    
    return chunks, embedding_matrix

In [7]:
raw_chunks = chunk_documents(raw_docs, chunk_size=500, overlap=50)
indexed_chunks, emb_mat = index_chunks(raw_chunks)

In [8]:
print(f"Loaded {len(raw_docs)} documents -> {len(indexed_chunks)} chunks | Matrix shape: {emb_mat.shape}")

Loaded 5 documents -> 19 chunks | Matrix shape: (19, 768)


In [9]:
indexed_chunks

[Chunk(source='company_handbook.pdf', chunk_id=0, text='ShopEase Philippines Employee Handbook\nVersion 1.0 - Effective 1 August 2026\n1. Company Overview\nShopEase Philippines is a fictional omnichannel retail company serving customers through online ordering and\nphysical stores across the Philippines. This handbook is designed as synthetic training material for an analytics\nand AI exercise.\n2. Working Hours\nCorporate office hours are 9:00 AM to 6:00 PM, Monday through Friday. Store employees follow schedules based\non branch operating hours and approve', embedding=array([-7.69941785e-02,  2.51770644e-02, -2.11463989e-01, -2.34837436e-02,
         9.91738334e-03, -1.85896470e-03,  1.43193392e-02, -5.10076598e-02,
         2.33822359e-02,  2.41770087e-02,  4.04158400e-02,  1.38054666e-02,
         3.68518794e-02,  5.31052812e-03,  2.54093496e-02, -9.03782070e-03,
         6.74494568e-03, -4.20817026e-02, -3.63622888e-02,  2.88205478e-02,
         4.61945070e-02, -2.57356250e-02, -4

In [10]:
emb_mat

array([[-0.07699418,  0.02517706, -0.21146399, ..., -0.01104095,
         0.00236118, -0.03414431],
       [-0.00121723,  0.01333297, -0.19120074, ..., -0.04529648,
        -0.02370448, -0.05741952],
       [ 0.00225391,  0.03657045, -0.18327419, ..., -0.06704346,
        -0.02684891, -0.00886953],
       ...,
       [ 0.01654478,  0.0347181 , -0.18669289, ..., -0.05195534,
        -0.02313997, -0.04612555],
       [ 0.00289424,  0.020623  , -0.19314217, ..., -0.0543918 ,
        -0.00971977, -0.06174654],
       [ 0.02633041, -0.00179211, -0.2101573 , ..., -0.04486471,
        -0.04406103, -0.06036459]], shape=(19, 768))

### 2.3 - Vector retrieval, grounded prompting, and question testing

In [11]:
def retrieve_top_k(
    question: str,
    chunks: list[Chunk] = indexed_chunks,
    embedding_matrix: np.ndarray = emb_mat,
    k: int = 3
) -> list[tuple[float, Chunk]]:
    """Retrieves top-k chunks via vectorized cosine similarity."""
    
    q_vec = get_embedding(question)
    q_norm = np.linalg.norm(q_vec)
    if q_norm > 0:
        q_vec = q_vec / q_norm

    # Dot product = cosine similarity because all vectors are unit-normalized
    scores = embedding_matrix @ q_vec # or np.matmul(A, B)
    top_indices = np.argsort(scores)[::-1][:k]
    return [(float(scores[idx]), chunks[idx]) for idx in top_indices]


def ask_rag(
    question: str,
    chat_model: str = CHAT_MODEL,
    k: int = 3
) -> tuple[str, list[tuple[float, Chunk]]]:
    """Answers a question strictly using retrieved policy chunks."""

    top_results = retrieve_top_k(question, k=k)
    context_blocks = [
        f"\n{c.text}"
        for score, c in top_results
    ]
    formatted_context = "\n\n---\n\n".join(context_blocks)

    prompt = (
        "Use ONLY the following context to answer the question.\n"
        "If the answer cannot be determined from the provided context, say:\n"
        '"That information is not available in the provided documents."\n\n'
        f"CONTEXT:\n{formatted_context}\n\n"
        f"QUESTION:\n{question}"
    )

    response = ollama.chat(
        model=chat_model, 
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response["message"]["content"], top_results


def run_rag_evaluation(questions: list[str]) -> None:
    """Displays Question -> Retrieved Sources/Chunks -> Generated Answer."""
    
    for i, q in enumerate(questions, 1):
        answer, hits = ask_rag(q)
        print(f"=== RAG Test {i} ===")
        print(f"QUESTION: {q}")
        print("---")
        print("RETRIEVED SOURCES:")
        for score, chunk in hits:
            preview = chunk.text.replace("\n", " ")[:110]
            print(f"  - [{chunk.source} | Chunk {chunk.chunk_id} | Score: {score:.3f}] {preview}...")
        print("---")
        print(f"GENERATED ANSWER:\n{answer}\n")

In [12]:
# Last question is out-of-scope grounding test
rag_test_questions = [
    "How many days does a customer have for a standard return?",
    "What should a customer provide when reporting a damaged item?",
    "Do Gold members receive free shipping?",
    "What does the company say about AI systems issuing refunds?",
    "Does ShopEase offer international shipping to Japan?"
]

In [13]:
run_rag_evaluation(rag_test_questions)

=== RAG Test 1 ===
QUESTION: How many days does a customer have for a standard return?
RETRIEVED SOURCES:
  - [return_refund_policy.docx | Chunk 2 | Score: 0.711] 4. Non-Returnable Items For hygiene and safety reasons, opened Personal Care items that directly contact the b...
  - [return_refund_policy.docx | Chunk 0 | Score: 0.702] ShopEase Philippines Return and Refund Policy Effective date: 1 August 2026 1. Standard Return Window Most eli...
  - [return_refund_policy.docx | Chunk 1 | Score: 0.668] d provide the order number and clear photos of the product and packaging. Eligible damaged-on-arrival items ma...
GENERATED ANSWER:
According to the provided context, a customer has 14 calendar days from the recorded delivery date to make a standard return.

=== RAG Test 2 ===
QUESTION: What should a customer provide when reporting a damaged item?
RETRIEVED SOURCES:
  - [company_handbook.pdf | Chunk 3 | Score: 0.714] Team. 7. Customer Data Customer records are confidential. Employees should

## 3 - Agent Tools (Structured knowledge)

In [15]:
def _parse_json_df(path: Path) -> pd.DataFrame:
    """Safely normalizes flat lists, wrapped lists, or ID-keyed JSON files into a DataFrame."""
    raw = json.loads(path.read_text(encoding="utf-8"))

    # Case 1: Already a flat list of dicts -> [{"customer_id": "CUST-001", ...}, ...]
    if isinstance(raw, list):
        return pd.DataFrame(raw)

    if isinstance(raw, dict):
        # Case 2: Wrapped list -> {"customers": [{"customer_id": "CUST-001", ...}]}
        if len(raw) == 1 and isinstance(next(iter(raw.values())), list):
            return pd.DataFrame(next(iter(raw.values())))

        # Case 3: ID-keyed dictionary -> {"CUST-001": {"name": ...}, ...}
        df = pd.DataFrame.from_dict(raw, orient="index")
        id_col = "customer_id" if "customer" in path.stem else f"{path.stem.rstrip('s')}_id"
        return df.reset_index(names=id_col)

    return pd.read_json(path)


def load_structured_tables(
    filenames: Iterable[str], 
    data_dir: str | Path = "./data"
) -> dict[str, pd.DataFrame]:
    """Loads structured datasets into a dictionary."""

    loaded: dict[str, pd.DataFrame] = {}
    for file in filenames:
        file_path = data_dir / file
        if not file_path.exists():
            print(f"[Warning] Missing dataset skipped: {file_path}")
            continue
        loader = TABLE_LOADERS.get(file_path.suffix.lower())
        if loader:
            loaded[file_path.stem] = loader(file_path)

    return loaded


TABLE_LOADERS: dict[str, Callable[[Path], pd.DataFrame]] = {
    ".csv": pd.read_csv,
    ".xlsx": pd.read_excel,
    ".json": _parse_json_df
}

In [16]:
structured_files = [
    "product_catalog.csv",
    "orders.xlsx",
    "customers.json",
    "stores.csv",
    "store_inventory.csv",
    "support_tickets.json",
]

tables = load_structured_tables(structured_files, data_dir=DATA_DIR)
tables.get("product_catalog")

,sku,product_name,category,unit_price_php,warranty_months,active,total_stock
0,SKU-0001,Astra Wireless Mouse,Electronics,699,6,Yes,99
1,SKU-0002,Vertex Mechanical Keyboard,Electronics,1199,0,Yes,22
2,SKU-0003,Nova USB-C Hub,Electronics,5999,12,Yes,80
3,SKU-0004,Astra Bluetooth Speaker,Electronics,499,3,Yes,16
4,SKU-0005,Kinetic Webcam,Electronics,1199,0,Yes,82
5,SKU-0006,Vertex Power Bank,Electronics,5999,24,Yes,96
6,SKU-0007,Vertex Wireless Earbuds,Electronics,2999,24,Yes,62
7,SKU-0008,Astra Portable SSD,Electronics,1299,3,Yes,102
8,SKU-0009,Nexa Laptop Stand,Electronics,2999,3,Yes,40
9,SKU-0010,Nexa Smart Plug,Electronics,999,0,Yes,18


In [17]:
def _lookup_one(df: pd.DataFrame, col: str, value: str, entity: str) -> dict:
    if col not in df.columns:
        return {"error": f"Column '{col}' not found in {entity} table. Available columns: {list(df.columns)}"}
    match = df[df[col] == value]
    if match.empty:
        return {"error": f"{entity} '{value}' not found."}
    return match.iloc[0].to_dict()


# Structured Data Tools
def get_customer(customer_id: str) -> dict:
    """Fetches customer profile details (name, email, membership_tier) by customer_id."""
    return _lookup_one(tables["customers"], "customer_id", customer_id, "Customer")


def get_order(order_id: str) -> dict:
    """Fetches order details (customer_id, sku, status, total_amount, delivery_date) by order_id."""
    return _lookup_one(tables["orders"], "order_id", order_id, "Order")


def get_product(sku: str) -> dict:
    """Fetches product master data (name, category, price, warranty_months) by sku."""
    return _lookup_one(tables["product_catalog"], "sku", sku, "Product")


def get_store(store_id: str) -> dict:
    """Fetches store location and contact details by store_id."""
    return _lookup_one(tables["stores"], "store_id", store_id, "Store")


def check_inventory(sku: str) -> list[dict]:
    """Returns stores and available quantities where the given sku is in stock."""
    inv = tables["store_inventory"]
    matches = inv[(inv["sku"] == sku) & (inv["stock_on_hand"] > 0)]
    return matches[["store_id", "stock_on_hand"]].to_dict(orient="records")


def get_support_tickets(customer_id: str | None = None, order_id: str | None = None) -> list[dict]:
    """Fetches support tickets matching customer_id and/or order_id."""
    tickets = tables["support_tickets"]
    mask = pd.Series(False, index=tickets.index)
    if customer_id:
        mask |= tickets["customer_id"] == customer_id
    if order_id:
        mask |= tickets["order_id"] == order_id
    return tickets[mask].to_dict(orient="records")


# RAG Knowledge Tool
def search_policy(question: str) -> str:
    """Searches ShopEase policy documents (returns, refunds, shipping, warranties, handbook) for rules."""
    top_results = retrieve_top_k(question, k=3)
    return "\n\n".join(
        f"\n{c.text}" for score, c in top_results
    )

**Unit tests:**

In [18]:
# Test get_order
pprint(get_order("ORD-1025"), sort_dicts=False)

{'order_id': 'ORD-1025',
 'order_date': Timestamp('2026-08-25 00:00:00'),
 'customer_id': 'CUST-001',
 'sku': 'SKU-0042',
 'quantity': 1,
 'unit_price_php': 799,
 'membership_discount_pct': 0.0,
 'net_amount_php': 799.0,
 'payment_method': 'GCash',
 'status': 'Delivered',
 'delivery_date': Timestamp('2026-08-28 00:00:00'),
 'fulfillment_store_id': 'STR001'}


In [19]:
# Test get_product
pprint(get_product("SKU-0042"), sort_dicts=False)

{'sku': 'SKU-0042',
 'product_name': 'Nexa Electric Toothbrush',
 'category': 'Personal Care',
 'unit_price_php': 799,
 'warranty_months': 3,
 'active': 'Yes',
 'total_stock': 102}


In [20]:
# Test check_inventory
pprint(check_inventory("SKU-0042"), sort_dicts=False)

[{'store_id': 'STR002', 'stock_on_hand': 7},
 {'store_id': 'STR005', 'stock_on_hand': 4}]


In [21]:
# Test search_policy
print(search_policy("damaged item return")[:150], "...")


d provide the order number and clear photos of the product and packaging. Eligible damaged-on-arrival items may be returned for replacement or refund ...


## 4 - AI Agent and multi-step reasoning

In [22]:
AGENT_TOOLS: list[Callable] = [
    get_customer,
    get_order,
    get_product,
    get_store,
    check_inventory,
    get_support_tickets,
    search_policy,
]

TOOL_REGISTRY: dict[str, Callable] = {fn.__name__: fn for fn in AGENT_TOOLS}

In [23]:
def run_agent(user_query: str, model: str = CHAT_MODEL, max_steps: int = 6) -> str:
    """Executes a bounded ReAct tool-calling loop with self-correcting error handling."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a ShopEase Philippines AI Agent. Use your tools to gather exact "
                "customer, order, product, inventory, ticket, and policy facts before answering. "
                "Only pass the exact arguments defined in each tool's signature."
            ),
        },
        {"role": "user", "content": user_query},
    ]

    for step in range(1, max_steps + 1):
        response = ollama.chat(model=model, messages=messages, tools=AGENT_TOOLS)
        msg = response["message"]
        messages.append(msg)

        tool_calls = msg.get("tool_calls")
        if not tool_calls:
            return msg["content"]

        for call in tool_calls:
            func_name = call["function"]["name"]
            args = call["function"]["arguments"]
            print(f"  [Step {step}] Tool Call -> {func_name}({args})")

            tool_fn = TOOL_REGISTRY.get(func_name)
            if not tool_fn:
                result = {"error": f"Unknown tool '{func_name}'. Available tools: {list(TOOL_REGISTRY.keys())}"}
            else:
                try:
                    result = tool_fn(**args)
                except TypeError as err:
                    valid_params = list(inspect.signature(tool_fn).parameters.keys())
                    result = {
                        "error": f"Invalid arguments for {func_name}: {err}. Pass ONLY these parameters: {valid_params}"
                    }

            messages.append({
                "role": "tool",
                "content": json.dumps(result, default=str),
            })

    return "Agent stopped: maximum reasoning steps reached."

In [24]:
# Test using a single tool
print(run_agent("What product is SKU-0042?"))

  [Step 1] Tool Call -> get_product({'sku': 'SKU-0042'})
The product with SKU-0042 is the Nexa Electric Toothbrush, which falls under the Personal Care category. It is currently priced at PHP 799 and has a warranty of 3 months. There are 102 units available in stock.


In [25]:
# Test using mutiple tools
multi_tool_queries = [
    "What product did CUST-001 purchase in ORD-1025?",
    "Does CUST-001's order ORD-1025 qualify for free shipping based on their membership and order amount?",
    "Is the product from ORD-1025 potentially still under warranty based on its delivery date and warranty period?"
]
for q in multi_tool_queries:
    print(f"\n--- Multi-Tool Query: {q} ---")
    print(run_agent(q))


--- Multi-Tool Query: What product did CUST-001 purchase in ORD-1025? ---
  [Step 1] Tool Call -> get_product({'sku': 'get_order', 'order_id': 'ORD-1025'})
  [Step 2] Tool Call -> get_product({'order_id': 'ORD-1025', 'sku': 'get_order'})
  [Step 3] Tool Call -> get_product({'order_id': 'ORD-1025', 'sku': 'PRODUCT-123'})
Since the `get_product` tool does not accept an `order_id` parameter, I will call another tool to get the product information for the given order.

{"name": "get_order_product", "parameters": {"order_id": "ORD-1025"}}

--- Multi-Tool Query: Does CUST-001's order ORD-1025 qualify for free shipping based on their membership and order amount? ---
  [Step 1] Tool Call -> search_policy({'question': "Does CUST-001's order ORD-1025 qualify for free shipping based on their membership and order amount?"})
Unfortunately, CUST-001 is a Silver member, so they do not qualify for free shipping based on their membership. However, their order ORD-1025 has a net merchandise amount of a

In [26]:
# Flagship hybrid agent + RAG scenario
hybrid_query = (
    "CUST-001 says the item from ORD-1025 arrived damaged. "
    "Based on the order and company policy, what should Customer Support tell the customer?"
)
print(f"\n--- Hybrid Agent + RAG Query ---\n{hybrid_query}")
print(run_agent(hybrid_query))


--- Hybrid Agent + RAG Query ---
CUST-001 says the item from ORD-1025 arrived damaged. Based on the order and company policy, what should Customer Support tell the customer?
  [Step 1] Tool Call -> search_policy({'question': 'item arrived damaged, company policy regarding damaged goods'})
"Hello, CUST-001. We apologize for the damaged item you received from our order ORD-1025. According to our company policy, you should provide us with clear photos of the item and packaging, and report the damage-on-arrival claim within 7 calendar days of delivery. Please also include your order number for our records. We will be happy to assist you with a replacement or refund for the damaged item, even if the package has been opened for inspection. If you have any further questions or concerns, please don't hesitate to reach out to us. Thank you for your patience and cooperation."


## 5 - Agent safety and governance

In [27]:
def issue_refund(order_id: str) -> dict:
    """SIMULATED WRITE TOOL: Demonstrates safe human-in-the-loop gating."""
    return {
        "status": "SIMULATION ONLY",
        "order_id": order_id,
        "message": "Refund execution blocked: requires human manager approval.",
    }

In [28]:
AGENT_TOOLS: list[Callable] = [
    get_customer,
    get_order,
    get_product,
    get_store,
    check_inventory,
    get_support_tickets,
    search_policy,
    issue_refund
]

TOOL_REGISTRY: dict[str, Callable] = {fn.__name__: fn for fn in AGENT_TOOLS}

In [29]:
refund_request = (
    "CUST-001 reported that order ORD-1025 arrived damaged (Ticket TKT-5001) and is asking for a refund. "
    "Check the order details, verify ShopEase's policy on damaged items and AI-issued refunds, "
    "and process the refund if allowed."
)

print(run_agent(refund_request))

  [Step 1] Tool Call -> get_order({'order_id': 'ORD-1025'})
  [Step 1] Tool Call -> search_policy({'question': 'damaged items refund'})
  [Step 1] Tool Call -> issue_refund({'order_id': 'ORD-1025'})
Dear CUST-001,

We apologize for the inconvenience and disappointment caused by the damaged item you received in your order ORD-1025. According to our Return and Refund Policy, you are eligible for a refund within 7 calendar days of delivery.

Our policy states that items that arrive damaged or defective should be reported to Customer Support within 7 calendar days of delivery. We have received your report (Ticket TKT-5001) and are processing your refund.

However, our system is currently set to require human manager approval for refund execution. This is a security measure to ensure that all refunds are processed correctly and in accordance with our policies.

Please wait for further instructions from our customer support team, who will be in touch with you shortly to assist you with the r

## 6 - Discussion

**1. What is the role of embeddings in the RAG application?**

Embeddings convert unstructured text chunks and user questions into dense numerical vectors so the system can measure semantic similarity mathematically rather than relying on exact keyword overlap.

**2. Why did we split documents into chunks?**

Chunking keeps prompts concise and ensures each embedding vector represents a focused topic (such as damaged-item returns), improving retrieval accuracy compared to averaging an entire multi-topic document.

**3. How does the system determine which chunks are relevant to a question?**

It embeds the user's question into the same vector space, computes cosine similarity against all stored chunk vectors, and selects the top-$k$ highest-scoring chunks.

**4. What is the difference between RAG and an Agent?**

RAG follows a fixed linear pipeline i.e. (Retrieve -> Augment -> Generate) to answer questions from documents, whereas an Agent dynamically reasons about a goal and chooses which tools (and how many steps) to execute.

**5. Why might structured data be better accessed through a tool instead of embedding every row?**

Structured tables rely on exact ID matches, aggregations, and frequently changing values (like live stock counts), which deterministic code/queries handle accurately without vector approximation errors.

**6. How can an Agent use RAG?**

By wrapping the vector retrieval function as a callable tool (e.g. `search_policy()`), allowing the Agent to query unstructured policy documents alongside structured database tools in a single workflow.

**7. What risks appear when an Agent is allowed to perform actions?**

Hallucinated parameters, unauthorized data exposure, repeated tool loops, or prompt-injection attacks can trigger unintended financial refunds, order cancellations, or corrupted inventory records.

**8. What was the most interesting thing I learned while building the application?**

- Underneat the capabilities of an Agent that looks intelligent to us human users is just a collection of algorithms working together.
- Being able to witness how an Agent chains discoveries across modalities through a structured lookup to discover factual figures and cross-referencing unstructured knowledge to form a grounded answer from data.